In [ ]:
import torch
import torchvision.models as models
import torch.utils.benchmark as benchmark

# 1. Initialize the model and data
device = "cuda" if torch.cuda.is_available() else "cpu"
model = models.resnet18().to(device)
input_data = torch.randn(32, 3, 224, 224).to(device)

# 2. Prepare the compiled model
compiled_model = torch.compile(model)

# 3. Warm up the compiled model (Crucial step!)
# The first call triggers the actual compilation process
with torch.no_grad():
    compiled_model(input_data)

# 4. Define the benchmark tasks
def run_eager():
    with torch.no_grad():
        return model(input_data)

def run_compiled():
    with torch.no_grad():
        return compiled_model(input_data)

# 5. Measure performance
t_eager = benchmark.Timer(
    stmt="run_eager()",
    globals={"run_eager": run_eager},
    label="Eager Mode"
)

t_compiled = benchmark.Timer(
    stmt="run_compiled()",
    globals={"run_compiled": run_compiled},
    label="Compiled Mode"
)

print(t_eager.blocked_autorange())
print(t_compiled.blocked_autorange())

Compiling model... (this may take a minute)
Eager Mode
  3.72 s
  1 measurement, 1 runs , 1 thread
Compiled Mode
  2.22 s
  1 measurement, 1 runs , 1 thread
